# Building the Borough-Year Dataset

# Borough-Year Feature Engineering

This notebook transforms the cleaned London transaction dataset into a borough-year level analytical dataset.

Each row represents one London borough in one calendar year, with aggregated market indicators that will later be used for predictive modelling.

## Workflow

Processed London Transactions

↓

Aggregate by Borough and Year

↓

Create Borough-Year Dataset

↓

Save Processed Dataset

## Output

The processed dataset will be saved as:

`data/processed/borough_year_dataset.csv`

In [57]:
import pandas as pd
from pathlib import Path

In [ ]:
# Project folders

RAW_DATA = Path("../data/raw")
PROCESSED_DATA = Path("../data/processed")

print("Raw folder:", RAW_DATA)
print("Processed folder:", PROCESSED_DATA)

In [ ]:
# Load the processed London dataset

DATA_PATH = PROCESSED_DATA / "london_transactions_2018_2025.csv"

london_transactions = pd.read_csv(DATA_PATH)

print(london_transactions.shape)

In [ ]:
london_transactions.head()

In [ ]:
london_transactions["District"].nunique()

In [ ]:
sorted(london_transactions["District"].unique())

In [63]:
borough_year = (
    london_transactions
    .groupby(["District", "Year"])
)

In [64]:
borough_year = (
    london_transactions
    .groupby(["District", "Year"])
    .agg(
        Transactions=("Price", "count"),
        Average_Price=("Price", "mean"),
        Median_Price=("Price", "median"),
        Min_Price=("Price", "min"),
        Max_Price=("Price", "max"),
        Price_STD=("Price", "std")
    )
    .reset_index()
)

In [ ]:
borough_year.head()

In [ ]:
borough_year.shape

In [ ]:
# Create a working copy for feature engineering

features_df = borough_year.copy()

features_df.head()

In [68]:
features_df = features_df.sort_values(
    by=["District", "Year"]
).reset_index(drop=True)

In [ ]:
features_df.head(15)

## Creating Growth Features

Historical price growth is calculated within each borough.

These features describe how the local housing market has evolved over time and provide temporal information for future prediction tasks.

In [70]:
features_df["Price_Growth"] = (
    features_df
    .groupby("District")["Average_Price"]
    .pct_change()
)

In [ ]:
features_df[
    [
        "District",
        "Year",
        "Average_Price",
        "Price_Growth"
    ]
].head(15)

In [72]:
features_df = features_df.rename(
    columns={
        "Price_Growth": "Average_Price_Growth"
    }
)

In [ ]:
features_df.columns

In [74]:
features_df["Median_Price_Growth"] = (
    features_df
    .groupby("District")["Median_Price"]
    .pct_change()
)

In [ ]:
features_df[
    [
        "District",
        "Year",
        "Average_Price_Growth",
        "Median_Price_Growth"
    ]
].head(15)

In [ ]:
features_df.columns

## Creating Prediction Targets

Prediction targets are generated by shifting the growth variables one year forward.

This allows each observation to describe the current market while predicting the following year's market performance.

In [77]:
features_df["Target_Average_Price_Growth"] = (
    features_df
    .groupby("District")["Average_Price_Growth"]
    .shift(-1)
)

In [ ]:
features_df[
    [
        "District",
        "Year",
        "Average_Price_Growth",
        "Target_Average_Price_Growth"
    ]
].head(15)

In [79]:
features_df["Target_Median_Price_Growth"] = (
    features_df
    .groupby("District")["Median_Price_Growth"]
    .shift(-1)
)

In [ ]:
features_df[
    [
        "District",
        "Year",
        "Median_Price_Growth",
        "Target_Median_Price_Growth"
    ]
].head(15)

In [ ]:
features_df.head()

## Data Quality Audit

Before saving the final borough-year feature dataset, a series of validation checks is performed.

These checks verify that the dataset structure, completeness and feature engineering process are correct. They also help detect unexpected changes if the pipeline is modified in the future.

In [ ]:
print("=" * 60)
print("DATA QUALITY AUDIT")
print("=" * 60)

print(f"Dataset shape: {features_df.shape}")

# Verify that all 33 London boroughs are present
print(f"\nNumber of boroughs: {features_df['District'].nunique()}")
assert features_df["District"].nunique() == 33

# Verify that all expected years are included
years = sorted(features_df["Year"].unique())
print(f"Years covered: {years}")

# Verify that each borough appears only once per year
duplicates = features_df.duplicated(
    subset=["District", "Year"]
).sum()

print(f"Duplicate Borough-Year pairs: {duplicates}")
assert duplicates == 0

# Display missing values for manual inspection
print("\nMissing values by column:")
print(features_df.isna().sum())

### Missing Values Interpretation

The missing values shown above are expected and do not indicate data quality issues.

- The first year (2018) has no historical data available to calculate year-over-year growth.
- The last year (2025) has no subsequent year available to generate prediction targets.

Therefore:

- `Average_Price_Growth` contains one missing value for each borough.
- `Median_Price_Growth` contains one missing value for each borough.
- `Target_Average_Price_Growth` contains one missing value for each borough.
- `Target_Median_Price_Growth` contains one missing value for each borough.

These missing values are a natural consequence of the feature engineering process and are expected.

## Validation Checks

Before using this dataset for predictive modelling, a small set of validation checks is performed to verify that the aggregation, feature engineering and target construction have been completed correctly.

In [ ]:
expected_rows = 33 * 8

print("Expected rows:", expected_rows)
print("Actual rows:", len(features_df))

In [ ]:
check = london_transactions[
    (london_transactions["District"] == "BARNET") &
    (london_transactions["Year"] == 2024)
]

print("Transactions:", len(check))
print("Average Price:", check["Price"].mean())
print("Median Price:", check["Price"].median())

In [ ]:
features_df[
    (features_df["District"] == "BARNET") &
    (features_df["Year"] == 2024)
]

In [ ]:
features_df[
    features_df["District"] == "BARNET"
][
    [
        "Year",
        "Average_Price_Growth",
        "Target_Average_Price_Growth"
    ]
]

## Validation Summary

Three validation checks were performed before finalising the feature dataset:

- The expected number of borough-year observations was confirmed.
- Aggregated statistics were manually verified against the raw transaction data.
- Prediction targets were validated to ensure correct one-year forward alignment.

The dataset passed all validation checks and is ready for predictive modelling.

In [ ]:
OUTPUT_FILE = PROCESSED_DATA / "borough_year_features.csv"

features_df.to_csv(OUTPUT_FILE, index=False)

print("Saved to:", OUTPUT_FILE)

In [ ]:
saved_df = pd.read_csv(OUTPUT_FILE)

print(saved_df.shape)
saved_df.head()